# Naive Unlearning — Sample-Wise Forgetting
### Master's Research: Machine Unlearning — Multi-Class Image Classification

**What this notebook does:**

Naive unlearning is the conceptually simplest unlearning method and serves as the **gold standard reference** in the unlearning literature. The idea is straightforward: remove the forget samples from the training set and retrain the model from scratch on the remaining data (the retain set). The resulting model provably never saw the forget samples, so it is the theoretically correct unlearned model.

It is called *naive* not because it is wrong, but because it is computationally expensive — retraining from scratch for every deletion request is impractical at scale. Every approximate unlearning method you implement later will be evaluated by how closely it matches this retrained model.

**Forget set options (controlled by config):**
- `random_samples` — a fixed number of randomly selected images from across all classes
- `single_class` — all (or a fixed number of) images from one specific class

**Evaluation outputs:**
- Retain accuracy — can the model still correctly classify the data it should remember?
- Forget accuracy — how well does the model predict the forget samples? (should drop)
- Test accuracy — overall generalisation on the held-out test set
- Side-by-side comparison table: Original model vs Naive retrained model

## 1. Imports

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import time

from utils import build_resnet18, evaluate, per_class_accuracy, \
                  load_checkpoint, save_checkpoint, STATS

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 2. Configuration

This is the single place you change to switch datasets, forget-set strategy, or forget-set size. Nothing below this cell needs to be modified for standard experiments.

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET    = "CIFAR10"       # "CIFAR10" | "CIFAR100"
DATA_ROOT  = "./data"
CHECKPOINT_DIR = "./checkpoints"

# Path to the original trained model (produced by 01_train.ipynb)
ORIGINAL_CKPT = os.path.join(CHECKPOINT_DIR,
                             f"resnet18_{DATASET.lower()}_best.pth")

# ── Forget set strategy ───────────────────────────────────────────────────────
#
# "random_samples" : randomly pick FORGET_SIZE images from the full training set
# "single_class"   : pick FORGET_SIZE images from CLASS_TO_FORGET only
#                    (set FORGET_SIZE = None to forget the entire class)
#
FORGET_STRATEGY = "random_samples"  # "random_samples" | "single_class"
FORGET_SIZE     = 500               # number of samples to forget
CLASS_TO_FORGET = 0                 # only used when FORGET_STRATEGY = "single_class"
                                    # (0 = 'airplane' for CIFAR-10)

# ── Retraining hyper-parameters ───────────────────────────────────────────────
# Keep identical to the original training for a fair comparison
BATCH_SIZE    = 128
NUM_EPOCHS    = 100
LR            = 0.1
MOMENTUM      = 0.9
WEIGHT_DECAY  = 5e-4
LR_MILESTONES = [50, 75]
LR_GAMMA      = 0.1

NUM_CLASSES = 10 if DATASET == "CIFAR10" else 100

print(f"Dataset         : {DATASET}")
print(f"Forget strategy : {FORGET_STRATEGY}")
if FORGET_STRATEGY == "single_class":
    size_str = str(FORGET_SIZE) if FORGET_SIZE else "entire class"
    print(f"Class to forget : {CLASS_TO_FORGET}  |  Size: {size_str}")
else:
    print(f"Forget size     : {FORGET_SIZE}")

## 3. Load Dataset & Build Forget / Retain / Test Splits

Three splits are constructed from the original data:

| Split | Contents | Used for |
|-------|----------|----------|
| **Forget set** $D_f$ | Samples to be unlearned | Measuring forget accuracy |
| **Retain set** $D_r$ | $D_{train} \setminus D_f$ | Retraining the naive model |
| **Test set** | Original held-out test split | Measuring generalisation |

In [ ]:
mean, std = STATS[DATASET]["mean"], STATS[DATASET]["std"]

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

DatasetClass = (torchvision.datasets.CIFAR10 if DATASET == "CIFAR10"
                else torchvision.datasets.CIFAR100)

# Full training set (with augmentation) — used to build retain loader
full_train = DatasetClass(DATA_ROOT, train=True,  download=True,
                          transform=train_transform)
# Full training set (no augmentation) — used to evaluate forget accuracy
full_train_eval = DatasetClass(DATA_ROOT, train=True, download=False,
                               transform=test_transform)
test_dataset    = DatasetClass(DATA_ROOT, train=False, download=False,
                               transform=test_transform)

# ── Build forget / retain index sets ─────────────────────────────────────────
all_targets = torch.tensor(full_train.targets)   # shape: (50000,)
all_indices = list(range(len(full_train)))

if FORGET_STRATEGY == "random_samples":
    # Randomly sample FORGET_SIZE indices from the entire training set
    forget_indices = sorted(random.sample(all_indices, FORGET_SIZE))

elif FORGET_STRATEGY == "single_class":
    # All indices whose label matches CLASS_TO_FORGET
    class_indices = (all_targets == CLASS_TO_FORGET).nonzero(as_tuple=True)[0].tolist()

    if FORGET_SIZE is None or FORGET_SIZE >= len(class_indices):
        # Forget the entire class
        forget_indices = class_indices
    else:
        # Forget a random subset of that class
        forget_indices = sorted(random.sample(class_indices, FORGET_SIZE))

else:
    raise ValueError(f"Unknown FORGET_STRATEGY: {FORGET_STRATEGY}")

# Retain indices = everything not in forget set
forget_set_idx = set(forget_indices)
retain_indices = [i for i in all_indices if i not in forget_set_idx]

print(f"Total training samples : {len(all_indices):,}")
print(f"Forget set size        : {len(forget_indices):,}")
print(f"Retain set size        : {len(retain_indices):,}")
print(f"Test set size          : {len(test_dataset):,}")

if FORGET_STRATEGY == "single_class":
    class_name = full_train.classes[CLASS_TO_FORGET]
    print(f"Forgetting class       : {CLASS_TO_FORGET} ({class_name})")

## 4. Build Data Loaders

In [ ]:
# Retain loader — used for retraining (has augmentation via full_train)
retain_loader = DataLoader(
    Subset(full_train, retain_indices),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True
)

# Forget loader — used for evaluation only (no augmentation, deterministic)
forget_loader = DataLoader(
    Subset(full_train_eval, forget_indices),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)

# Retain eval loader — same data as retain_loader but no augmentation
retain_eval_loader = DataLoader(
    Subset(full_train_eval, retain_indices),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)

# Test loader
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)

print("Data loaders ready.")
print(f"  Retain train batches : {len(retain_loader)}")
print(f"  Forget eval batches  : {len(forget_loader)}")
print(f"  Test batches         : {len(test_loader)}")

### Why two versions of the retain set?

- `retain_loader` uses augmentation (random crop, flip) because it is used **during retraining** — augmentation helps the model generalise from the smaller retain set.
- `retain_eval_loader` uses no augmentation because it is used **during evaluation** — you want deterministic, consistent measurements, not results that vary depending on which random crop was applied.

## 5. Evaluate the Original Model (Baseline)

Before doing anything, record what the original model scores on all three splits. These numbers are the baseline everything else is compared against.

In [ ]:
criterion = nn.CrossEntropyLoss()

print("Loading original model...")
original_model = load_checkpoint(ORIGINAL_CKPT, DEVICE)

print("\nEvaluating original model on all splits...")
orig_retain_loss, orig_retain_acc = evaluate(original_model, retain_eval_loader,
                                              criterion, DEVICE)
orig_forget_loss, orig_forget_acc = evaluate(original_model, forget_loader,
                                              criterion, DEVICE)
orig_test_loss,   orig_test_acc   = evaluate(original_model, test_loader,
                                              criterion, DEVICE)

print(f"\n{'Split':<15} {'Loss':>8}  {'Accuracy':>9}")
print("-" * 36)
print(f"{'Retain':<15} {orig_retain_loss:>8.4f}  {orig_retain_acc:>8.2f}%")
print(f"{'Forget':<15} {orig_forget_loss:>8.4f}  {orig_forget_acc:>8.2f}%")
print(f"{'Test':<15} {orig_test_loss:>8.4f}  {orig_test_acc:>8.2f}%")

## 6. Naive Unlearning — Retrain from Scratch on Retain Set

A new model is initialised with **random weights** and trained only on $D_r$. It never sees $D_f$ at any point during training.

In [ ]:
# ── Fresh model ───────────────────────────────────────────────────────────────
naive_model = build_resnet18(NUM_CLASSES).to(DEVICE)

naive_optimizer = optim.SGD(
    naive_model.parameters(),
    lr=LR, momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    nesterov=True,
)
naive_scheduler = optim.lr_scheduler.MultiStepLR(
    naive_optimizer,
    milestones=LR_MILESTONES,
    gamma=LR_GAMMA,
)

print(f"Retraining on retain set ({len(retain_indices):,} samples) for {NUM_EPOCHS} epochs...\n")

naive_history = {"train_loss": [], "train_acc": [],
                 "test_loss":  [], "test_acc":  []}

best_naive_acc  = 0.0
naive_best_ckpt = os.path.join(
    CHECKPOINT_DIR,
    f"resnet18_{DATASET.lower()}_naive_unlearn_best.pth"
)

print(f"{'Epoch':>6}  {'LR':>8}  "
      f"{'Retain Loss':>11}  {'Retain Acc':>10}  "
      f"{'Test Loss':>9}  {'Test Acc':>8}  {'Time':>6}")
print("-" * 78)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    naive_model.train()

    # ── One training epoch ────────────────────────────────────────────────────
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in retain_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        naive_optimizer.zero_grad()
        outputs = naive_model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        naive_optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct    += outputs.argmax(1).eq(labels).sum().item()
        total      += images.size(0)

    tr_loss = total_loss / total
    tr_acc  = 100.0 * correct / total

    # ── Evaluate on test set ──────────────────────────────────────────────────
    te_loss, te_acc = evaluate(naive_model, test_loader, criterion, DEVICE)
    naive_scheduler.step()

    naive_history["train_loss"].append(tr_loss)
    naive_history["train_acc"].append(tr_acc)
    naive_history["test_loss"].append(te_loss)
    naive_history["test_acc"].append(te_acc)

    current_lr = naive_scheduler.get_last_lr()[0]
    elapsed    = time.time() - t0

    print(f"{epoch:>6}  {current_lr:>8.5f}  "
          f"{tr_loss:>11.4f}  {tr_acc:>9.2f}%  "
          f"{te_loss:>9.4f}  {te_acc:>7.2f}%  {elapsed:>5.1f}s")

    if te_acc > best_naive_acc:
        best_naive_acc = te_acc
        save_checkpoint(
            naive_model, naive_best_ckpt,
            epoch=epoch, test_acc=te_acc,
            dataset=DATASET, num_classes=NUM_CLASSES,
            extra={
                "unlearning_method": "naive_retrain",
                "forget_strategy":   FORGET_STRATEGY,
                "forget_size":       len(forget_indices),
                "retain_size":       len(retain_indices),
            }
        )

print(f"\nRetraining complete. Best test accuracy: {best_naive_acc:.2f}%")

## 7. Evaluate the Naive Unlearned Model

In [ ]:
print("Loading best naive unlearned model...")
naive_model = load_checkpoint(naive_best_ckpt, DEVICE)

print("\nEvaluating on all splits...")
naive_retain_loss, naive_retain_acc = evaluate(naive_model, retain_eval_loader,
                                                criterion, DEVICE)
naive_forget_loss, naive_forget_acc = evaluate(naive_model, forget_loader,
                                                criterion, DEVICE)
naive_test_loss,   naive_test_acc   = evaluate(naive_model, test_loader,
                                                criterion, DEVICE)

print(f"\n{'Split':<15} {'Loss':>8}  {'Accuracy':>9}")
print("-" * 36)
print(f"{'Retain':<15} {naive_retain_loss:>8.4f}  {naive_retain_acc:>8.2f}%")
print(f"{'Forget':<15} {naive_forget_loss:>8.4f}  {naive_forget_acc:>8.2f}%")
print(f"{'Test':<15} {naive_test_loss:>8.4f}  {naive_test_acc:>8.2f}%")

## 8. Side-by-Side Comparison

This is the central result of naive unlearning. The key things to observe:

- **Retain accuracy** should stay close between the two models — the naive model still learned from 49,500 samples
- **Forget accuracy** is the most important metric — a drop here means the model has genuinely unlearned those samples
- **Test accuracy** should remain similar — unlearning should not destroy generalisation

In [ ]:
print("=" * 62)
print(f"{'Metric':<20} {'Original':>12}  {'Naive Retrain':>14}  {'Δ':>6}")
print("=" * 62)

rows = [
    ("Retain Accuracy",  orig_retain_acc, naive_retain_acc),
    ("Forget Accuracy",  orig_forget_acc, naive_forget_acc),
    ("Test Accuracy",    orig_test_acc,   naive_test_acc),
]

for name, orig, naive in rows:
    delta = naive - orig
    sign  = "+" if delta >= 0 else ""
    print(f"{name:<20} {orig:>11.2f}%  {naive:>13.2f}%  {sign}{delta:>5.2f}%")

print("=" * 62)

## 9. Training Curves — Naive Retrained Model

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, naive_history["train_loss"], label="Retain (train)", linewidth=2)
axes[0].plot(epochs, naive_history["test_loss"],  label="Test",           linewidth=2)
for m in LR_MILESTONES:
    axes[0].axvline(x=m, color="grey", linestyle="--", alpha=0.5)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].set_title("Naive Retrain — Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, naive_history["train_acc"], label="Retain (train)", linewidth=2)
axes[1].plot(epochs, naive_history["test_acc"],  label="Test",           linewidth=2)
for m in LR_MILESTONES:
    axes[1].axvline(x=m, color="grey", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title(f"Naive Retrain — Accuracy (best test: {best_naive_acc:.2f}%)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f"{DATASET} — Naive Unlearning ({FORGET_STRATEGY}, {len(forget_indices)} samples)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(
    os.path.join(CHECKPOINT_DIR,
                 f"resnet18_{DATASET.lower()}_naive_unlearn_curves.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()

## 10. Forget Accuracy Visualised

Plot the original model's predicted confidence on a sample of forget-set images, alongside the naive model's predictions. A well-unlearned model should show lower confidence and more uniform distributions on the forget samples.

In [ ]:
@torch.no_grad()
def get_confidences(model, loader, device, n_samples=16):
    """
    Return predicted class, true label, and max softmax confidence
    for the first n_samples images in the loader.
    """
    model.eval()
    images_out, labels_out, preds_out, confs_out = [], [], [], []

    for images, labels in loader:
        images = images.to(device)
        probs  = torch.softmax(model(images), dim=1)
        confs, preds = probs.max(dim=1)

        images_out.append(images.cpu())
        labels_out.append(labels)
        preds_out.append(preds.cpu())
        confs_out.append(confs.cpu())

        if sum(len(x) for x in labels_out) >= n_samples:
            break

    images_out = torch.cat(images_out)[:n_samples]
    labels_out = torch.cat(labels_out)[:n_samples]
    preds_out  = torch.cat(preds_out)[:n_samples]
    confs_out  = torch.cat(confs_out)[:n_samples]
    return images_out, labels_out, preds_out, confs_out


def denorm(img_tensor, mean, std):
    mean_t = torch.tensor(mean).view(3, 1, 1)
    std_t  = torch.tensor(std).view(3, 1, 1)
    return (img_tensor * std_t + mean_t).permute(1, 2, 0).numpy().clip(0, 1)


N_VIS = 8
orig_imgs, orig_true, orig_pred, orig_conf = get_confidences(
    original_model, forget_loader, DEVICE, N_VIS)
naiv_imgs, naiv_true, naiv_pred, naiv_conf = get_confidences(
    naive_model, forget_loader, DEVICE, N_VIS)

classes = full_train.classes
fig, axes = plt.subplots(2, N_VIS, figsize=(N_VIS * 2, 5))

for i in range(N_VIS):
    # Row 0 — original model predictions on forget samples
    axes[0, i].imshow(denorm(orig_imgs[i], mean, std))
    axes[0, i].set_title(
        f"True: {classes[orig_true[i]]}\n"
        f"Pred: {classes[orig_pred[i]]}\n"
        f"Conf: {orig_conf[i]:.2f}",
        fontsize=7
    )
    axes[0, i].axis("off")

    # Row 1 — naive unlearned model predictions on same forget samples
    axes[1, i].imshow(denorm(naiv_imgs[i], mean, std))
    axes[1, i].set_title(
        f"True: {classes[naiv_true[i]]}\n"
        f"Pred: {classes[naiv_pred[i]]}\n"
        f"Conf: {naiv_conf[i]:.2f}",
        fontsize=7
    )
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original",      fontsize=10, rotation=90, labelpad=40)
axes[1, 0].set_ylabel("Naive Retrain", fontsize=10, rotation=90, labelpad=40)

plt.suptitle("Forget-set predictions: Original vs Naive Retrained model",
             fontsize=12)
plt.tight_layout()
plt.show()

## 11. Summary

| Item | Value |
|------|-------|
| Dataset | `DATASET` |
| Forget strategy | `FORGET_STRATEGY` |
| Forget set size | `len(forget_indices)` |
| Retain set size | `len(retain_indices)` |
| Unlearning method | Naive retraining from scratch |
| Role in thesis | **Gold standard / reference model** |

**Interpretation guide:**

- If forget accuracy of the naive model is significantly **lower** than the original model → unlearning has worked at the sample level
- If retain and test accuracy remain close → the model has not lost general knowledge
- The naive model's forget accuracy is the **target** for approximate methods: an approximate unlearning method is considered successful if its forget accuracy is close to this number while being much cheaper to compute than full retraining

**Why forget accuracy may not drop to chance level:**

For random-sample forgetting, the 500 forgotten samples are not a representative subset of their classes — the retain set still contains thousands of other images from the same classes. The retrained model learns those classes perfectly well from the retain data, so it will still correctly classify the forgotten samples. This is expected and correct. The forget accuracy drop is more pronounced in class-wise forgetting, where the model has no retain data for the forgotten class.